# Deliverable 1: Kafka Ingestion & Dead-Letter Topic (DLQ) Quarantine
**Course**: Modern Data Engineering for AI Systems Capstone (SDAIA Academy)

This notebook demonstrates:
1. Ingestion boundary schema enforcement using Pydantic data contracts.
2. Publishing valid records alongside intentional malformed records.
3. Routing valid records to raw/bronze storage and malformed records to Dead-Letter Topic / Quarantine zone with rejection reasons recorded.

In [ ]:
import os
import sys
import json
from pprint import pprint

# Ensure src directory is in path
sys.path.append('..')

from src.ingestion.producer import publish_events
from src.ingestion.consumer import process_ingestion

## 1. Publish Raw Dataset to Kafka Topic (with Malformed Records)

In [ ]:
pub_results = publish_events()
print(f"Producer Run Status: {pub_results['status']}")
print(f"Total Events Generated: {pub_results['total_events']}")
print(f"Kafka Connected: {pub_results['kafka_connected']}")

## 2. Ingest, Validate Contracts, & Route to Dead-Letter Topic (DLQ)

In [ ]:
ingest_results = process_ingestion()
print(f"Total Ingested: {ingest_results['total_processed']}")
print(f"  |-- Validated Events: {ingest_results['valid_count']}")
print(f"  |-- Quarantined DLQ Events: {ingest_results['quarantine_count']}")

## 3. Inspect Quarantined DLQ Records (Failure Path Proof)

In [ ]:
with open(ingest_results['quarantine_path'], 'r', encoding='utf-8') as f:
    dlq_data = json.load(f)

print(f"[PROOF OF FAILURE PATH] Quarantined Records in Dead-Letter Topic:")
for rec in dlq_data:
    print(f"  - Quarantine ID : {rec['quarantine_id']}")
    print(f"    Rejection Reason: {rec['rejection_reason']}")
    print(f"    Failed Field    : {rec['failed_field']}")
    print(f"    Raw Article ID  : {rec['raw_payload'].get('article_id')}\n")